# Fantasy World Predictions — LOTR Universe
## Step 3: Preprocessing + training pipeline

---

### Goal of this notebook
Turn `lotr_characters_clean.csv` into **one saved classification model per candidate target**, using a shared preprocessing factory. The Streamlit app will later load these artifacts — it will not train on the fly, and it will never mix LOTR with other universes.

### Design decisions (locked for LOTR)

**(a) Pre-train every candidate target now, with a flexible factory**

The factory is: *drop `name` + the chosen target, treat the rest as features, build a sklearn `Pipeline`*. We call that factory once per target and **save** the fitted pipeline. That matches the app goal ("trained model predicts") while still supporting flexible target selection later.

Training at click-time would also work on 765 rows, but it would make the end-user wait and would hide evaluation. Pre-trained `.pkl` files keep the UI dumb and fast.

**(b) Stratified splits, always, for LOTR classification**

`race` and `realm` are imbalanced (Dragons = 5, Shire = 5, Unknown realm = 570). A random split can put a whole rare class on one side. We use `stratify=y` on an 80/20 split, `class_weight='balanced'` in both models, and **macro-F1** as the model-picking metric (accuracy would just reward "always Male" / "always Unknown").

`name` is an ID, never a feature or a target. All five remaining columns are classification targets; there is no regression column in this clean table.

### Candidate targets
| Target | Why |
|---|---|
| `gender` | Binary, usable |
| `race` | Multi-class, imbalanced |
| `is_dead` | Engineered binary |
| `has_spouse` | Engineered binary |
| `realm` | Optional **weak** target (mostly `Unknown`) — trained and saved, but flagged |

### What we are *not* doing here
Harry Potter / Star Wars / GoT, the Streamlit app, and heavy hyperparameter search. Light defaults + a fair comparison is enough to freeze LOTR artifacts.

## Section 1: Imports & load

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
TEST_SIZE = 0.20

base_dirs = [Path.cwd(), *Path.cwd().parents]
ROOT = next((d for d in base_dirs if (d / 'data' / 'lotr_characters_clean.csv').exists()), Path.cwd())
DATA_DIR = ROOT / 'data'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_DIR / 'lotr_characters_clean.csv')
print(f'Loaded {df.shape[0]} rows × {df.shape[1]} columns from {DATA_DIR / "lotr_characters_clean.csv"}')
print('Nulls:', int(df.isna().sum().sum()))
df.head()

Loaded 765 rows × 8 columns from C:\Users\hp\Downloads\fantasy-ml-project\fantasy-ml-project\data\lotr_characters_clean.csv
Nulls: 0


,gender,name,race,realm,is_dead,has_spouse,hair_group,birth_age
0,Female,Adanel,Men,Unknown,0,1,Unknown,Unknown
1,Male,Boromir,Men,Unknown,1,0,Dark,Third Age
2,Male,Lagduf,Orcs,Unknown,1,0,Unknown,Unknown
3,Male,Tarcil,Men,Arnor,1,1,Unknown,Third Age
4,Male,Fire-drake of Gondolin,Other,Unknown,0,0,Unknown,Unknown


## Section 2: Class balance (why stratification matters)

If a class has at least 2 rows, sklearn can stratify an 80/20 split. LOTR's rarest classes after cleaning are 5 rows, so we are safe. We still report **macro-F1** so a 74% majority class cannot look like a good model.

In [2]:
ID_COL = 'name'
CANDIDATE_TARGETS = ['gender', 'race', 'is_dead', 'has_spouse', 'realm']
WEAK_TARGETS = {'realm'}  # 78% Unknown in EDA; kept as optional

assert df[ID_COL].is_unique, 'name should be a unique ID'

for col in CANDIDATE_TARGETS:
    counts = df[col].value_counts()
    print(f'\n=== {col}  ({counts.size} classes, min={counts.min()}, majority={counts.max() / len(df):.1%}) ===')
    print(counts.to_string())


=== gender  (2 classes, min=134, majority=82.5%) ===
gender
Male      631
Female    134

=== race  (9 classes, min=5, majority=50.5%) ===
race
Men           386
Hobbits       142
Elves         103
Other          42
Dwarves        42
Ainur          28
Orcs           10
Half-elven      7
Dragons         5

=== is_dead  (2 classes, min=210, majority=72.5%) ===
is_dead
1    555
0    210

=== has_spouse  (2 classes, min=315, majority=58.8%) ===
has_spouse
1    450
0    315

=== realm  (10 classes, min=5, majority=74.5%) ===
realm
Unknown            570
Other               73
Gondor              33
NÃºmenor            28
Rohan               20
Arthedain           14
Arnor               11
Valinor              6
Shire                5
Lonely Mountain      5


## Section 3: The reusable factory

For a chosen target `T`:

1. `y = df[T]`, `X = df.drop(columns=['name', T])`
2. Categorical columns → `OneHotEncoder(handle_unknown='ignore')` so a rare realm in the test fold does not crash inference
3. Numeric columns (`is_dead`, `has_spouse` when they are features) → passthrough
4. Wrap encoder + classifier in one `Pipeline` so the app stores **one object** per target

We train two classifiers on the same split:
- **Logistic regression** — linear baseline, `class_weight='balanced'`
- **Random forest** — non-linear, same class weight

The winner (higher test macro-F1) is what we save.

In [3]:
def split_xy(frame, target):
    y = frame[target]
    X = frame.drop(columns=[ID_COL, target])
    return X, y


def feature_groups(X):
    cat_cols = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    num_cols = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    return cat_cols, num_cols


def make_preprocessor(cat_cols, num_cols):
    transformers = []
    if cat_cols:
        transformers.append(
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        )
    if num_cols:
        transformers.append(('num', 'passthrough', num_cols))
    if not transformers:
        raise ValueError('No feature columns left after dropping the target')
    return ColumnTransformer(transformers, remainder='drop')


def make_pipeline(preprocessor, estimator):
    # clone() so each target gets its own fitted classifier
    return Pipeline([
        ('prep', preprocessor),
        ('clf', clone(estimator)),
    ])


def metrics_dict(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }


CANDIDATE_MODELS = {
    'logreg': LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    'random_forest': RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

## Section 4: Train / evaluate one target

Split **once** per target (same fold for both models), print a classification report for the winner, and draw a confusion matrix.

In [4]:
def train_one_target(frame, target, show_plots=True):
    X, y = split_xy(frame, target)
    cat_cols, num_cols = feature_groups(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
    )

    rows = []
    fitted = {}
    for name, estimator in CANDIDATE_MODELS.items():
        pipe = make_pipeline(make_preprocessor(cat_cols, num_cols), estimator)
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        m = metrics_dict(y_test, pred)
        m.update({'target': target, 'model': name})
        rows.append(m)
        fitted[name] = (pipe, pred)

    scores = pd.DataFrame(rows).sort_values('f1_macro', ascending=False).reset_index(drop=True)
    winner_name = scores.loc[0, 'model']
    winner_pipe, winner_pred = fitted[winner_name]

    print(f'\n########  target = {target}  |  features = {list(X.columns)}  ########')
    print(f'categorical: {cat_cols}  numeric: {num_cols}')
    print(f'train={len(X_train)}  test={len(X_test)}  stratified=True')
    if target in WEAK_TARGETS:
        print('⚠ weak target — high missingness before fill; treat app results as illustrative')
    print(scores.to_string(index=False))
    print(f'\nWinner: {winner_name}')
    print(classification_report(y_test, winner_pred, zero_division=0))

    if show_plots:
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_test, winner_pred, ax=ax, xticks_rotation=45, colorbar=False,
        )
        ax.set_title(f'{target} — {winner_name}')
        fig.tight_layout()
        plt.show()

    artifact = {
        'universe': 'lotr',
        'target': target,
        'task': 'classification',
        'weak_target': target in WEAK_TARGETS,
        'model_name': winner_name,
        'pipeline': winner_pipe,
        'feature_columns': list(X.columns),
        'categorical_columns': cat_cols,
        'numeric_columns': num_cols,
        'classes': [str(c) for c in winner_pipe.classes_],
        'metrics': scores.loc[0, ['accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted']].to_dict(),
        'all_model_metrics': scores.to_dict(orient='records'),
    }
    return artifact, scores

## Section 5: Run every candidate target

This is the loop the app is designed around: **same code, different `target` string.**

In [5]:
artifacts = {}
comparison_frames = []

for target in CANDIDATE_TARGETS:
    artifact, scores = train_one_target(df, target, show_plots=True)
    artifacts[target] = artifact
    comparison_frames.append(scores)

comparison = pd.concat(comparison_frames, ignore_index=True)
print('\n=== All targets × models (sorted by macro-F1) ===')
comparison.sort_values(['target', 'f1_macro'], ascending=[True, False])


########  target = gender  |  features = ['race', 'realm', 'is_dead', 'has_spouse', 'hair_group', 'birth_age']  ########
categorical: ['race', 'realm', 'hair_group', 'birth_age']  numeric: ['is_dead', 'has_spouse']
train=612  test=153  stratified=True
 accuracy  balanced_accuracy  f1_macro  f1_weighted target         model
 0.686275           0.649471  0.593805     0.719209 gender random_forest
 0.627451           0.642857  0.559746     0.671459 gender        logreg

Winner: random_forest
              precision    recall  f1-score   support

      Female       0.30      0.59      0.40        27
        Male       0.89      0.71      0.79       126

    accuracy                           0.69       153
   macro avg       0.60      0.65      0.59       153
weighted avg       0.79      0.69      0.72       153



C:\Users\hp\AppData\Local\Temp\ipykernel_11084\3210777735.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



########  target = race  |  features = ['gender', 'realm', 'is_dead', 'has_spouse', 'hair_group', 'birth_age']  ########
categorical: ['gender', 'realm', 'hair_group', 'birth_age']  numeric: ['is_dead', 'has_spouse']
train=612  test=153  stratified=True
 accuracy  balanced_accuracy  f1_macro  f1_weighted target         model
 0.555556           0.441198  0.369528     0.613314   race        logreg
 0.588235           0.425144  0.350225     0.624353   race random_forest

Winner: logreg
              precision    recall  f1-score   support

       Ainur       0.30      1.00      0.46         6
     Dragons       0.00      0.00      0.00         1
     Dwarves       0.27      0.75      0.40         8
       Elves       0.80      0.57      0.67        21
  Half-elven       0.00      0.00      0.00         2
     Hobbits       0.67      0.64      0.65        28
         Men       1.00      0.51      0.67        77
        Orcs       0.00      0.00      0.00         2
       Other       0.44

C:\Users\hp\AppData\Local\Temp\ipykernel_11084\3210777735.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



########  target = is_dead  |  features = ['gender', 'race', 'realm', 'has_spouse', 'hair_group', 'birth_age']  ########
categorical: ['gender', 'race', 'realm', 'hair_group', 'birth_age']  numeric: ['has_spouse']
train=612  test=153  stratified=True
 accuracy  balanced_accuracy  f1_macro  f1_weighted  target         model
 0.738562           0.694015  0.685185     0.743646 is_dead random_forest
 0.692810           0.655084  0.641257     0.702588 is_dead        logreg

Winner: random_forest
              precision    recall  f1-score   support

           0       0.52      0.60      0.56        42
           1       0.84      0.79      0.81       111

    accuracy                           0.74       153
   macro avg       0.68      0.69      0.69       153
weighted avg       0.75      0.74      0.74       153



...


########  target = has_spouse  |  features = ['gender', 'race', 'realm', 'is_dead', 'hair_group', 'birth_age']  ########
categorical: ['gender', 'race', 'realm', 'hair_group', 'birth_age']  numeric: ['is_dead']
train=612  test=153  stratified=True
 accuracy  balanced_accuracy  f1_macro  f1_weighted     target         model
 0.705882           0.695238  0.695691     0.705518 has_spouse random_forest
 0.653595           0.662698  0.652109     0.656121 has_spouse        logreg

Winner: random_forest
              precision    recall  f1-score   support

           0       0.65      0.63      0.64        63
           1       0.75      0.76      0.75        90

    accuracy                           0.71       153
   macro avg       0.70      0.70      0.70       153
weighted avg       0.71      0.71      0.71       153



C:\Users\hp\AppData\Local\Temp\ipykernel_11084\3210777735.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



########  target = realm  |  features = ['gender', 'race', 'is_dead', 'has_spouse', 'hair_group', 'birth_age']  ########
categorical: ['gender', 'race', 'hair_group', 'birth_age']  numeric: ['is_dead', 'has_spouse']
train=612  test=153  stratified=True
⚠ weak target — high missingness before fill; treat app results as illustrative
 accuracy  balanced_accuracy  f1_macro  f1_weighted target         model
 0.490196           0.450789  0.307289     0.551353  realm random_forest
 0.300654           0.511053  0.198810     0.326992  realm        logreg

Winner: random_forest
                 precision    recall  f1-score   support

          Arnor       1.00      0.50      0.67         2
      Arthedain       0.11      0.67      0.18         3
         Gondor       0.00      0.00      0.00         7
Lonely Mountain       0.00      0.00      0.00         1
       NÃºmenor       0.38      1.00      0.56         5
          Other       0.27      0.60      0.38        15
          Rohan       0.

C:\Users\hp\AppData\Local\Temp\ipykernel_11084\3210777735.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,accuracy,balanced_accuracy,f1_macro,f1_weighted,target,model
0,0.686275,0.649471,0.593805,0.719209,gender,random_forest
1,0.627451,0.642857,0.559746,0.671459,gender,logreg
6,0.705882,0.695238,0.695691,0.705518,has_spouse,random_forest
7,0.653595,0.662698,0.652109,0.656121,has_spouse,logreg
4,0.738562,0.694015,0.685185,0.743646,is_dead,random_forest
5,0.692810,0.655084,0.641257,0.702588,is_dead,logreg
2,0.555556,0.441198,0.369528,0.613314,race,logreg
3,0.588235,0.425144,0.350225,0.624353,race,random_forest
8,0.490196,0.450789,0.307289,0.551353,realm,random_forest
9,0.300654,0.511053,0.198810,0.326992,realm,logreg


## Section 6: Save winner pipelines + a small manifest

Each `models/lotr_<target>.pkl` is a dict the app can `joblib.load`:
`pipeline`, `feature_columns`, `classes`, `task`, `metrics`.

`models/lotr_manifest.json` lists every target so Streamlit can populate the dropdown without scanning pickle internals.

In [6]:
manifest = {
    'universe': 'lotr',
    'n_rows': int(len(df)),
    'id_column': ID_COL,
    'targets': {},
}

for target, artifact in artifacts.items():
    path = MODELS_DIR / f'lotr_{target}.pkl'
    joblib.dump(artifact, path)
    print(f'Saved {path.name}  ({artifact["model_name"]}, f1_macro={artifact["metrics"]["f1_macro"]:.3f})')
    manifest['targets'][target] = {
        'task': artifact['task'],
        'weak_target': artifact['weak_target'],
        'model_file': path.name,
        'model_name': artifact['model_name'],
        'feature_columns': artifact['feature_columns'],
        'classes': artifact['classes'],
        'metrics': {k: float(v) for k, v in artifact['metrics'].items()},
    }

manifest_path = MODELS_DIR / 'lotr_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(f'\nManifest → {manifest_path}')

Saved lotr_gender.pkl  (random_forest, f1_macro=0.594)
Saved lotr_race.pkl  (logreg, f1_macro=0.370)


Saved lotr_is_dead.pkl  (random_forest, f1_macro=0.685)
Saved lotr_has_spouse.pkl  (random_forest, f1_macro=0.696)


Saved lotr_realm.pkl  (random_forest, f1_macro=0.307)

Manifest → C:\Users\hp\Downloads\fantasy-ml-project\fantasy-ml-project\models\lotr_manifest.json


## Section 7: Smoke test — reload and predict one row

Confirms the pickle is what the app will call: load artifact, pass a one-row DataFrame with the stored feature columns.

In [7]:
sample_target = 'is_dead'
loaded = joblib.load(MODELS_DIR / f'lotr_{sample_target}.pkl')
row = df.drop(columns=[ID_COL, sample_target]).iloc[[0]]
pred = loaded['pipeline'].predict(row)[0]
proba = loaded['pipeline'].predict_proba(row)[0]
print('features:', row.to_dict(orient='records')[0])
print(f'predicted {sample_target} =', pred)
print('probabilities:', dict(zip(loaded['classes'], [round(float(p), 3) for p in proba])))

features: {'gender': 'Female', 'race': 'Men', 'realm': 'Unknown', 'has_spouse': 1, 'hair_group': 'Unknown', 'birth_age': 'Unknown'}
predicted is_dead = 0
probabilities: {'0': 0.99, '1': 0.01}


---
## What's next?

LOTR models are on disk. Options, in order:

1. **Read the metrics** — if `realm` is near-useless, we can hide it in the app or keep it labeled weak.
2. **Optional light tuning** on the winner (still LOTR-only) if macro-F1 looks poor on `race`.
3. **Harry Potter EDA** — same skeleton as LOTR, new schema, still no app.

> Tell your assistant what you want next, or flag a target whose metrics you want improved before we leave LOTR.